In [1]:
# Gensim 라이브러리
# 대규모 한국어 코퍼스(Wikipedia, Naver News등) 미리학습  Word2Vec, FastText 모델파일 이 들어 있는 라이브러리

In [2]:
# step1 : 사전학습된 모델로드
import gensim
# FastText 모델 로드
# OOV(Out-Of-Vocabulary, 사전에 없는 단어  ) 문제에 강점이 있다 오타가많은 리뷰데이터 특화

# step2 : 임베딩 행렬 구축
# 로드한 모델 수십만개의 단어 벡터가 있음, 우리가 구성한 단어장(vocab)에 있는 단어만 필요(단어장 크기, 임베딩 차원)크기의 빈행렬생성
# 우리 단어장에 있는 단어가 사전학습한 모델에 존재하면 그 벡터를 복사

# step3 : nn.Embedding에 가중치 덮어쓰기

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}
df = pd.DataFrame(data)

def tokenizer(text) : return text.split()

vocab = {'<PAD>':0, '<UNK>':1}
for review in df['reviews']:
    for word in tokenizer(review):
        if word not in vocab: vocab[word] = len(vocab)

max_sequence_len = 8
def text_to_sequence(text,vocab,max_sequence_len):
    seq = [vocab.get(word,vocab['<UNK>']) for word in tokenizer(text)]
    return seq + [vocab['<PAD>']]*(max_sequence_len-len(seq)) \
                if len(seq) < max_sequence_len \
                    else seq[:max_sequence_len]
df['sequences'] = df['reviews'].apply(lambda x : text_to_sequence(x,vocab,max_sequence_len))

class ReviewDataSet(Dataset):
    def __init__(self,sequence,labels):
        self.x = torch.LongTensor(sequence)
        self.y = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        return self.x[index], self.y[index]
dataloader = DataLoader(ReviewDataSet(df['sequences'].tolist(), df['ratings'].tolist()),
                        batch_size=2,shuffle=True
                        )    

https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz

In [4]:
import urllib
import gzip
import shutil
import os

print('파일 다운로드....')
url = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ko.300.vec.gz'
gz_filename = 'cc.ko.300.vec.gz'
vec_filename = 'cc.ko.300.vec'

urllib.request.urlretrieve(url, gz_filename)

print('압축해제......')
with gzip.open(gz_filename,'rb') as f_in:
    with open(vec_filename,'wb') as f_out:
        shutil.copyfileobj(f_in,f_out)

os.remove(gz_filename)
print('압축해제 및 파일정리 완료')

파일 다운로드....
압축해제......
압축해제 및 파일정리 완료


In [ ]:
# 모델 로드(로컬 FastText) & 임베딩 행렬 구축
import gensim
VOCAB_SIZE = len(vocab)
EMBED_DIM = 300  # FastText가 300차원
MODEL_PATH = vec_filename
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(MODEL_PATH,binary=False, unicode_errors='ignore')

KeyboardInterrupt: 

In [ ]:
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
oov_count = 0

for word, idx in vocab.items():
    if idx == 0: continue # <PAD>는 0 유지
    
    if word in w2v_model:
        embedding_matrix[idx] = w2v_model[word]
    else:
        # 사전에 없는 단어는 정규분포를 따르는 난수로 초기화
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(EMBED_DIM, ))
        oov_count += 1
        
print(f"-> 모델 로드 완료! (우리 단어장 단어: {VOCAB_SIZE}개 / 사전에 없는 단어: {oov_count}개)")
pretrained_weight = torch.tensor(embedding_matrix, dtype=torch.float32)

# ==========================================
# 4. TextCNN 모델 정의
# ==========================================
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, filter_sizes, num_filters, pretrained_weight=None, freeze_emb=False):
        super(TextCNN, self).__init__()
        
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        # FastText 가중치 덮어씌우기
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)
            self.embedding.weight.requires_grad = not freeze_emb

        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embed_dim)) for fs in filter_sizes
        ])
        
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(1)
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x)).squeeze(3)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled_outputs.append(p)
            
        x_cat = self.dropout(torch.cat(pooled_outputs, dim=1))
        return self.fc(x_cat).squeeze(1)

# ==========================================
# 5. 모델 학습 (Training)
# ==========================================
FILTER_SIZES = [2, 3, 4] 
NUM_FILTERS = 100        

model = TextCNN(
    vocab_size=VOCAB_SIZE, 
    embed_dim=EMBED_DIM, 
    filter_sizes=FILTER_SIZES, 
    num_filters=NUM_FILTERS, 
    pretrained_weight=pretrained_weight, 
    freeze_emb=False # 학습하며 임베딩도 우리 데이터에 맞게 미세조정
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

print("\n--- 본격적인 학습을 시작합니다 ---")
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    model.train() 
    
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

print("--- 학습 완료! ---")